# Data to Decisions Project

In [1]:
# Import Packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns


In [2]:
# Create timestamp and author
from datetime import datetime
time_now=datetime.now().strftime('%m_%d_%Y_%I%M%p')
print("Current Time:{}".format(time_now))
print("Caleb Manea")

Current Time:11_04_2026_1208PM
Caleb Manea


In [3]:
df = pd.read_csv("meridian_finance_monthly_finances.csv")

In [4]:
print("ORIGINAL DATA:")
print(df.head())

ORIGINAL DATA:
     month           department_name   account_category     actual     budget  \
0  2026-01  General & Administrative  Operating Expense  407943.85  100363.95   
1  2026-02  General & Administrative  Operating Expense  385825.05   98581.59   
2  2026-03  General & Administrative  Operating Expense  357855.01   99184.53   
3  2026-04  General & Administrative  Operating Expense  269644.76   98270.37   
4  2026-05  General & Administrative  Operating Expense  500666.33   98636.37   

   dollar_variance  percent_variance  
0        307579.90            306.46  
1        287243.46            291.38  
2        258670.48            260.80  
3        171374.39            174.39  
4        402029.96            407.59  


In [5]:
# 3 largest dolar variances
df["absolute_dollar_variance"] = df["dollar_variance"].abs()

top3_dollars = df.nlargest(3, "absolute_dollar_variance")

print("TOP 3 LARGEST DOLLAR VARIANCES:")
print(top3_dollars[[
    "month",
    "department_name",
    "account_category",
    "dollar_variance"
]].to_string(index=False))

TOP 3 LARGEST DOLLAR VARIANCES:
  month        department_name  account_category  dollar_variance
2026-05      Sales & Marketing           Revenue       -670090.32
2026-07      Sales & Marketing           Revenue       -569936.01
2026-06 Research & Development Operating Expense        511413.02


In [6]:
# 3 largest percentage variances
df["absolute_percent_variance"] = df["percent_variance"].abs()

top3_percent = df.nlargest(3, "absolute_percent_variance")

print("\nTOP 3 LARGEST PERCENTAGE VARIANCES:")
print(top3_percent[[
    "month",
    "department_name",
    "account_category",
    "percent_variance"
]].to_string(index=False))


TOP 3 LARGEST PERCENTAGE VARIANCES:
  month        department_name  account_category  percent_variance
2026-05      Sales & Marketing           Revenue           -837.64
2026-07      Sales & Marketing           Revenue           -728.70
2026-06 Research & Development Operating Expense            644.31


In [7]:
# Flag variacnes over 15% AND over $10,000
df["materiality_flag"] = (
    (df["absolute_percent_variance"] > 15) &
    (df["absolute_dollar_variance"] > 10000)
)

In [8]:
# Display only flagged items 
flagged = df[df["materiality_flag"]].copy()

print("MATERIALITY FLAGS:")
print(flagged[["month", "department_name", "account_category", "dollar_variance", "percent_variance", "materiality_flag"]].to_string(index=False))

print("Total flagged records:", len(flagged))

MATERIALITY FLAGS:
  month          department_name  account_category  dollar_variance  percent_variance  materiality_flag
2026-01 General & Administrative Operating Expense        307579.90            306.46              True
2026-02 General & Administrative Operating Expense        287243.46            291.38              True
2026-03 General & Administrative Operating Expense        258670.48            260.80              True
2026-04 General & Administrative Operating Expense        171374.39            174.39              True
2026-05 General & Administrative Operating Expense        402029.96            407.59              True
2026-06 General & Administrative Operating Expense        265542.02            265.98              True
2026-07 General & Administrative Operating Expense        211695.27            216.78              True
2026-02               Operations              COGS        -49780.44            -26.47              True
2026-03               Operations             

In [17]:
# Group records by department and account
group_cols = ["department_name", "account_category"]

# Count total months for each department/account
total_months = (df.groupby(group_cols)["month"].nunique().reset_index(name="total_months"))

# Count flagged months for each department/account
flagged_months = (flagged.groupby(group_cols)["month"].nunique().reset_index(name="flagged_months"))

# Combine the results
trend_summary = total_months.merge(flagged_months,on=group_cols,how="left")

trend_summary["flagged_months"] = (trend_summary["flagged_months"].fillna(0).astype(int))

# Calculate frequency of flagged months
trend_summary["flagged_percentage"] = (trend_summary["flagged_months"] /trend_summary["total_months"] * 100)

# Classify as persistent if flagged in
# more than half of the months
def classify_trend(row):
    if row["flagged_months"] > row["total_months"] / 2:
        return "Persistent Trend"
    elif row["flagged_months"] > 0:
        return "One-Time Spike"
    else:
        return "Not Flagged"

trend_summary["classification"] = (trend_summary.apply(classify_trend, axis=1))

print("MONTHLY TREND CLASSIFICATION:")
print(trend_summary.to_string(index=False))
# 0 "one-time spikes" under classification rule

MONTHLY TREND CLASSIFICATION:
         department_name  account_category  total_months  flagged_months  flagged_percentage   classification
General & Administrative Operating Expense             7               7          100.000000 Persistent Trend
              Operations              COGS             7               4           57.142857 Persistent Trend
              Operations Operating Expense             7               6           85.714286 Persistent Trend
  Research & Development Operating Expense             7               7          100.000000 Persistent Trend
       Sales & Marketing Operating Expense             7               7          100.000000 Persistent Trend
       Sales & Marketing           Revenue             7               7          100.000000 Persistent Trend
